The ``AAclust().pre_select_scales()`` method narrows an amino acid scales DataFrame by **AAontology metadata only** — it drops the columns whose ``category`` is in ``cat_out`` or whose ``subcategory`` is in ``subcat_out``, based on ``df_cat``. No clustering happens here: it is the preparation step before a redundancy reduction with ``select_scales`` (correlation threshold) or ``filter_coverage`` (subcategory coverage).

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False

# Load amino acid scales (rows = amino acids, columns = scale IDs)
df_scales = aa.load_scales()
aa.display_df(df_scales, n_rows=10, show_shape=True)

DataFrame shape: (20, 586)


The exclusion is driven by the scale-category table (``df_cat``), which maps every scale to its AAontology ``category`` and ``subcategory``. When ``df_cat`` is omitted, the bundled table is loaded automatically.

In [2]:
df_cat = aa.load_scales(name="scales_cat")
aa.display_df(df_cat, n_rows=10, show_shape=True)

DataFrame shape: (586, 5)


,scale_id,category,subcategory,scale_name,scale_description
1,LINS030110,ASA/Volume,Accessible surface area (ASA),ASA (folded coil/turn),"Total median ac...s et al., 2003)"
2,LINS030113,ASA/Volume,Accessible surface area (ASA),ASA (folded coil/turn),"% total accessi...s et al., 2003)"
3,JANJ780101,ASA/Volume,Accessible surface area (ASA),ASA (folded protein),"Average accessi...n et al., 1978)"
4,JANJ780103,ASA/Volume,Accessible surface area (ASA),ASA (folded protein),"Percentage of e...n et al., 1978)"
5,LINS030104,ASA/Volume,Accessible surface area (ASA),ASA (folded protein),"Total median ac...s et al., 2003)"
6,LINS030107,ASA/Volume,Accessible surface area (ASA),ASA (folded protein),"% total accessi...s et al., 2003)"
7,CHOC760102,ASA/Volume,Accessible surface area (ASA),ASA (folded proteins),"Residue accessi...(Chothia, 1976)"
8,LINS030116,ASA/Volume,Accessible surface area (ASA),ASA (folded β-strand),"Total median ac...s et al., 2003)"
9,LINS030119,ASA/Volume,Accessible surface area (ASA),ASA (folded β-strand),"% total accessi...s et al., 2003)"
10,LINS030103,ASA/Volume,Accessible surface area (ASA),Hydrophilic ASA,"Hydrophilic acc...s et al., 2003)"


Pass ``cat_out`` to drop one or more whole AAontology categories. Here we remove the ``Composition`` category; the returned DataFrame keeps the amino acid index and only the surviving scale columns, in their original order:

In [3]:
aac = aa.AAclust()
df_no_comp = aac.pre_select_scales(df_scales, df_cat=df_cat, cat_out=["Composition"])
print("All scales:", df_scales.shape[1], "-> without 'Composition':", df_no_comp.shape[1])
aa.display_df(df_no_comp, n_rows=10, show_shape=True)

All scales: 586 -> without 'Composition': 528
DataFrame shape: (20, 528)


Use ``subcat_out`` to drop individual subcategories (a finer unit than a category). A single name may be given as a string or a list:

In [4]:
df_no_subcat = aac.pre_select_scales(df_scales, df_cat=df_cat,
                                     subcat_out=["Hydrophilicity", "β-turn"])
print("Without the two subcategories:", df_no_subcat.shape[1])
kept_subcats = df_cat[df_cat["scale_id"].isin(df_no_subcat.columns)]["subcategory"]
print("'Hydrophilicity' still present:", "Hydrophilicity" in set(kept_subcats))

Without the two subcategories: 537
'Hydrophilicity' still present: False


``cat_out`` and ``subcat_out`` combine — a scale is dropped if it matches either. This yields a study-specific, more interpretable scale set:

In [5]:
df_clf = aa.load_scales(unclassified_out=True)
subcat_out = ["β-turn", "β-turn (C-term)", "β-turn (N-term)", "Hydrophilicity"]
df_pre = aac.pre_select_scales(df_clf, df_cat=df_cat,
                               cat_out=["Composition"], subcat_out=subcat_out)
print("Classified scales:", df_clf.shape[1], "-> pre-selected:", df_pre.shape[1])
aa.display_df(df_pre, n_rows=10, show_shape=True)

Classified scales: 532 -> pre-selected: 415


DataFrame shape: (20, 415)


``pre_select_scales`` is the metadata step; the result composes with a redundancy reduction. For example, reduce the pre-selected set to one representative per subcategory cluster at 100% coverage with ``filter_coverage``:

In [6]:
ids = list(df_pre.columns)
names_ref = df_cat[df_cat["scale_id"].isin(ids)]["subcategory"].to_list()
selected_ids = aac.filter_coverage(X=df_pre.T, scale_ids=ids, names_ref=names_ref,
                                   min_coverage=100, df_cat=df_cat)
n_subcat = df_cat[df_cat["scale_id"].isin(selected_ids)]["subcategory"].nunique()
print(f"Coverage-reduced: {len(selected_ids)} scales covering {n_subcat} subcategories")

Coverage-reduced: 150 scales covering 52 subcategories
